In [41]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler, RobustScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
import seaborn as sns
import matplotlib.pyplot as plt

In [42]:
# Загрузка данных
data = pd.read_csv('data/etalon_df.csv', index_col=0)
# data = pd.read_csv('data/pat_etalon_df.csv')
data.drop(columns='study_ID', inplace=True)
# data.drop(columns='ID истории болезни', inplace=True)
data

,gender,age,rbc,hgb,hct,mcv,mchc,rdw,rdv_sd,ret_abs,...,pdw,plcr,soe,myelo,yunye,blasty,normoblast,normobl_abs,prolym,promyelo
0,0,87,2.21000,84.000000,26.300000,119.100000,320.00000,21.300000,52.144544,36.245919,...,16.438954,0.0,12.000000,1.0,2.0,0.0,0.0,0.0,0.0,0.0
1,0,87,1.91000,74.000000,23.600000,123.800000,315.00000,12.280473,41.106126,33.459317,...,16.905172,0.0,3.174857,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,0,87,4.54908,132.748598,44.744731,81.555698,336.94953,20.500000,43.022613,44.313909,...,15.528221,0.0,3.486072,1.0,1.0,0.0,0.0,0.0,0.0,0.0
3,0,87,2.97000,107.000000,32.500000,109.400000,329.00000,22.200000,38.790873,43.477298,...,15.175393,0.0,16.000000,0.0,2.0,0.0,0.0,0.0,0.0,0.0
5,1,51,7.29000,146.000000,48.500000,66.500000,301.00000,13.692463,40.780142,24.427060,...,15.951140,0.0,4.625649,0.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3110,1,71,3.15000,88.000000,26.000000,83.000000,338.00000,11.984088,57.000000,38.423377,...,15.138585,0.0,3.110324,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3111,1,71,3.17000,94.000000,26.000000,81.000000,366.00000,11.750663,54.000000,28.051271,...,16.490303,0.0,7.517509,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3112,1,71,3.81000,108.000000,31.000000,81.000000,350.00000,11.557387,54.000000,38.610999,...,15.675992,0.0,5.832216,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3113,1,71,4.04000,113.000000,33.000000,81.000000,346.00000,11.997245,54.000000,37.951416,...,15.702718,0.0,4.414060,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [43]:
data.info()

<class 'pandas.core.frame.DataFrame'>
Index: 1924 entries, 0 to 3114
Data columns (total 38 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   gender        1924 non-null   int64  
 1   age           1924 non-null   int64  
 2   rbc           1924 non-null   float64
 3   hgb           1924 non-null   float64
 4   hct           1924 non-null   float64
 5   mcv           1924 non-null   float64
 6   mchc          1924 non-null   float64
 7   rdw           1924 non-null   float64
 8   rdv_sd        1924 non-null   float64
 9   ret_abs       1924 non-null   float64
 10  cp            1924 non-null   float64
 11  anisocytos    1924 non-null   float64
 12  hypochromia   1924 non-null   float64
 13  macrocytos    1924 non-null   float64
 14  microcytos    1924 non-null   float64
 15  poikilocytos  1924 non-null   float64
 16  wbc           1924 non-null   float64
 17  ne_abs        1924 non-null   float64
 18  ly_abs        1924 non-null   flo

In [44]:
def check_data_quality(data,
                       freq_treshold=0.95,
                       nunique_threshold=0.95,
                       null_threshold=0):
    """Проводит поиск дублей, пропусков и неинформативных признаков и выводит результат в консоль.
    
    Неинформативным считается признак с долей уникальных значений или повторов выше установленного порога.
    
    Parameters
    ----------
        data : DataFrame
            Датафрейм для анализа
        freq_treshold : float
            Порог масимальной частоты встречаемости признака
        nunique_treshhold : float
            Порог максимальной уникальности признака
        null_threshold : float
            Порог максимального отстутсвия признака
    """
    
    # Поиск дублей
    try:
        print(f'Число найденных дублей: {
            data.duplicated().value_counts().loc[True]
            }\n')
    except KeyError:
        print('Количество дублей: 0\n')

    # Поиск пропущенных значений
    cols_null_percent = data.isnull().mean()
    cols_with_null = cols_null_percent[cols_null_percent > null_threshold]\
        .sort_values(ascending=False)
    if cols_with_null.shape[0]: display(cols_with_null)
    print(f'Количество признаков с пустыми значениями: {cols_with_null.shape[0]}\n')

    # Поиск неинформативных признаков
    low_information_cols = []
    bad_feat_flag = False
    # цикл по всем столбцам
    for col in data.columns:
        #наибольшая относительная частота в признаке
        top_freq = data[col].value_counts(normalize=True).max()
        #доля уникальных значений от размера признака
        nunique_ratio = data[col].nunique() / data[col].count()
        # сравниваем наибольшую частоту с порогом
        if top_freq > freq_treshold:
            bad_feat_flag = True
            low_information_cols.append(col)
            print(f'{col}: {top_freq:.2%} одинаковых значений')
        # сравниваем долю уникальных значений с порогом
        if nunique_ratio > nunique_threshold:
            bad_feat_flag = True
            low_information_cols.append(col)
            print(f'{col}: {nunique_ratio:.2%} уникальных значений')
    if not bad_feat_flag:
        print('Неинформативных признаков не найдено.')

Результат оценки: Нет данных для этой возрастной группы


In [76]:
# Референсные значения показателей ОАК по возрасту и полу
blood_test_references = {
    # Гемоглобин (Hb, г/л)
    "hgb": {
        "6-12_years": {
            "male": {"min": 120, "max": 160},
            "female": {"min": 120, "max": 150}
        },
        "12-18_years": {
            "male": {"min": 120, "max": 160},
            "female": {"min": 120, "max": 150}
        },
        "18-45_years": {
            "male": {"min": 117, "max": 166},
            "female": {"min": 117, "max": 153}
        },
        "45-65_years": {
            "male": {"min": 131, "max": 172},
            "female": {"min": 117, "max": 155}
        },
        "65+_years": {
            "male": {"min": 126, "max": 174},
            "female": {"min": 117, "max": 160}
        }
    },
    # Эритроциты
    "rbc": {
        "6-12_years": {
            "male": {"min": 3.8, "max": 4.8},
            "female": {"min": 3.8, "max": 4.8}
        },
        "12-18_years": {
            "male": {"min": 3.8, "max": 4.8},
            "female": {"min": 3.8, "max": 4.8}
        },
        "18-45_years": {
            "male": {"min": 4.3, "max": 5.7},
            "female": {"min": 3.8, "max": 5.1}
        },
        "45-65_years": {
            "male": {"min": 4.2, "max": 5.6},
            "female": {"min": 3.8, "max": 5.3}
        },
        "65+_years": {
            "male": {"min": 3.8, "max": 5.8},
            "female": {"min": 3.8, "max": 5.2}
        }
    }
}

def get_ref(gender, age, parameter):
    # Определение возрастной группы
    age_group = None
    if age <= 1/12:  # 0-1 месяц
        age_group = "0-1_month"
    elif age <= 1:    # 1-12 месяцев
        age_group = "1-12_months"
    elif age <= 6:    # 1-6 лет
        age_group = "1-6_years"
    elif age <= 12:   # 6-12 лет
        age_group = "6-12_years"
    elif age <= 18:   # 12-18 лет
        age_group = "12-18_years"
    elif age <= 45:   # 18-45 лет
        age_group = "18-45_years"
    elif age <= 65:   # 45-65 лет
        age_group = "45-65_years"
    else:             # 65+ лет
        age_group = "65+_years"

    gen = {0: 'female', 1: 'male'}

    ref = blood_test_references.get(parameter)[age_group][gen[gender]]
    return ref

# Вычисляем референсы один раз
refs = data.apply(lambda r: {
    'hgb_min': get_ref(r['gender'], r['age'], 'hgb')['min'],
    'rbc_min': get_ref(r['gender'], r['age'], 'rbc')['min'],
    'rbc_max': get_ref(r['gender'], r['age'], 'rbc')['max']
}, axis=1)

# Создаем маску
mask = (
    (data['hgb'] < refs.apply(lambda x: x['hgb_min'])) &
    (data['rbc'] >= refs.apply(lambda x: x['rbc_min'])) &
    (data['rbc'] <= refs.apply(lambda x: x['rbc_max']))
)

# data.loc[mask, 'class'] = 0
data.loc[mask,]

,gender,age,rbc,hgb,hct,mcv,mchc,rdw,rdv_sd,ret_abs,...,plcr,soe,myelo,yunye,blasty,normoblast,normobl_abs,prolym,promyelo,class
41,0,86,3.98,116.0,32.0,80.0,364.0,12.548519,48.000000,48.967843,...,0.0,5.638638,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
49,1,77,4.00,111.0,31.0,77.0,345.0,14.051848,70.000000,36.278576,...,0.0,4.784712,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
106,1,71,4.06,117.0,31.0,76.0,377.0,13.037508,50.000000,58.999907,...,0.0,3.635856,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
143,0,59,4.21,116.0,38.3,90.9,302.0,12.490645,37.308109,20.752447,...,0.0,7.515294,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
144,0,59,4.27,113.0,39.4,92.4,286.0,19.300000,50.476888,21.038259,...,0.0,6.786023,2.0,5.0,1.0,0.0,2.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3097,1,70,3.98,105.0,31.0,77.9,339.0,12.188794,51.953225,49.647906,...,0.0,7.180761,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3103,1,70,4.13,109.0,32.1,77.7,340.0,11.505976,50.900000,61.547280,...,0.0,5.607090,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3112,1,71,3.81,108.0,31.0,81.0,350.0,11.557387,54.000000,38.610999,...,0.0,5.832216,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3113,1,71,4.04,113.0,33.0,81.0,346.0,11.997245,54.000000,37.951416,...,0.0,4.414060,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


# определим возможные диагнозы и классы для них


## 0 - острая анемия
Острая анемия (быстро развивающееся снижение гемоглобина и эритроцитов) в клиническом анализе крови проявляется следующими изменениями:  

### **Основные признаки:**  
1. **Снижение гемоглобина (Hb)** – ниже нормы (у женщин < 120 г/л, у мужчин < 130 г/л, при острой кровопотере может падать резко).  
2. **Снижение эритроцитов (RBC)** – уменьшение количества красных кровяных клеток (норма: ♀ 3,7–4,7 × 10¹²/л, ♂ 4,0–5,0 × 10¹²/л).  
3. **Нормальный или слегка сниженный средний объем эритроцитов (MCV)** – чаще нормоцитарная анемия (MCV 80–100 фл), так как при острой кровопотере теряются зрелые клетки.  
4. **Нормальное или повышенное содержание гемоглобина в эритроцитах (MCH, MCHC)** – цветовой показатель обычно в норме (0,85–1,05).  

### **Дополнительные признаки (зависят от причины и стадии):**  
- **Ретикулоцитоз** – через 3–5 дней после кровопотери увеличиваются молодые формы эритроцитов (компенсаторная реакция костного мозга).  
- **Лейкоцитоз** – возможен при кровопотере (реактивный ответ на стресс).  
- **Тромбоцитоз** – реактивное увеличение тромбоцитов в ответ на кровопотерю.  
- **Снижение гематокрита (Ht)** – уменьшение объема эритроцитов в крови (норма: ♀ 36–46%, ♂ 40–48%).  

### **Причины острой анемии:**  
- Острая кровопотеря (травма, желудочно-кишечное кровотечение, маточное кровотечение).  
- Гемолиз (разрушение эритроцитов при инфекциях, токсинах, аутоиммунных процессах).  

Если анемия развилась быстро, в первые часы гемоглобин может быть в норме (из-за сгущения крови), но затем падает. Для уточнения причины нужен осмотр врача и дополнительные исследования (ферритин, билирубин, анализ на скрытую кровь и др.).


In [ ]:
gender, age = 0, 32

ac_anemia = (
    data['rbc'] < get_ref(gender, age, 'rbc') &
    data['hgb'] < get_ref(gender, age, 'hgb') &
    data['mcv'] == get_ref(gender, age, 'mcv') &
    data['mchc'] == get_ref(gender, age, 'mchc') &
    data['soe'] == get_ref(gender, age, 'soe') &
    data['htc'] < get_ref(gender, age, 'htc') &
    data['wbc'] == get_ref(gender, age, 'wbc') &
    data['ne_abs'] == get_ref(gender, age, 'ne_abs') &
    data['ba_abs'] == get_ref(gender, age, 'ba_abs') &
    data['eo_abs'] == get_ref(gender, age, 'eo_abs') &
    data['ly_abs'] == get_ref(gender, age, 'ly_abs') &
    data['plt'] > get_ref(gender, age, 'plt')
)

## 1 - хроническая анемия
Хроническая анемия в клиническом анализе крови (ОАК) проявляется следующими изменениями:  

### **1. Снижение уровня гемоглобина (Hb)**  
- **У мужчин**: < 130 г/л  
- **У женщин**: < 120 г/л  
- **У беременных**: < 110 г/л  

### **2. Снижение количества эритроцитов (RBC)**  
- **У мужчин**: < 4,0 × 10¹²/л  
- **У женщин**: < 3,5 × 10¹²/л  

### **3. Изменение эритроцитарных индексов**  
- **Средний объем эритроцитов (MCV)**:  
  - **Микроцитарная анемия** (железодефицит, талассемия): MCV < 80 фл  
  - **Нормоцитарная анемия** (хронические болезни, гемолиз): MCV 80–100 фл  
  - **Макроцитарная анемия** (В12-/фолиеводефицит): MCV > 100 фл  
- **Среднее содержание гемоглобина в эритроците (MCH)**: снижено при железодефиците (< 27 пг)  
- **Средняя концентрация гемоглобина в эритроците (MCHC)**: снижена при гипохромных анемиях (< 320 г/л)  

### **4. Изменения в мазке крови**  
- **Гипохромия** (бледные эритроциты) – при железодефиците  
- **Анизоцитоз, пойкилоцитоз** – при длительной анемии  
- **Микроциты** (железодефицит), **макроциты** (В12-дефицит)  
- **Шизоциты** – при гемолизе  

### **5. Другие возможные изменения**  
- **Ретикулоциты**:  
  - **Снижены** при гипопролиферативных анемиях (железодефицит, хронические болезни)  
  - **Повышены** при гемолизе или кровопотере  
- **Ферритин** (не входит в ОАК, но важен для диагностики):  
  - Снижен при железодефиците  
  - Повышен при анемии хронических заболеваний  

### **6. Сопутствующие изменения**  
- **Лейкопения/тромбоцитопения** – при В12-дефиците, апластической анемии  
- **Тромбоцитоз** – при хронической кровопотере  

### **Вывод**  
Признаки хронической анемии в ОАК зависят от ее типа, но ключевые критерии – **снижение Hb, эритроцитов и изменение эритроцитарных индексов**. Для уточнения причины нужны дополнительные анализы (ферритин, витамин В12, фолиевая кислота, CRP, биохимия).  

Если у вас есть конкретные цифры анализа, можно уточнить тип анемии.


In [ ]:
chr_anemia = (
    data['rbc'] < get_ref(gender, age, 'rbc') &
    data['hgb'] < get_ref(gender, age, 'hgb') &
    data['mcv'] < get_ref(gender, age, 'mcv') &
    data['mchc'] < get_ref(gender, age, 'mchc') &
    data['soe'] == get_ref(gender, age, 'soe') &
    data['htc'] == get_ref(gender, age, 'htc') &
    data['wbc'] == get_ref(gender, age, 'wbc') &
    data['ne_abs'] == get_ref(gender, age, 'ne_abs') &
    data['ba_abs'] == get_ref(gender, age, 'ba_abs') &
    data['eo_abs'] == get_ref(gender, age, 'eo_abs') &
    data['ly_abs'] == get_ref(gender, age, 'ly_abs') &
    data['plt'] > get_ref(gender, age, 'plt')
)

## 2 - острое воспаление
В клиническом анализе крови (ОАК) при остром воспалении наблюдаются следующие изменения:  

### **1. Лейкоцитоз** – повышение общего количества лейкоцитов (WBC):  
   - **Норма**: 4–9 × 10⁹/л  
   - **При воспалении**: > 10 × 10⁹/л (может достигать 15–20 × 10⁹/л и выше при гнойных процессах)  

### **2. Сдвиг лейкоцитарной формулы влево** – увеличение доли незрелых форм нейтрофилов:  
   - **Палочкоядерные нейтрофилы** ↑ (норма 1–6%, при воспалении > 10%)  
   - **Метамиелоциты (юные)** и **миелоциты** (в норме отсутствуют, появляются при тяжелом воспалении)  

### **3. Ускорение СОЭ** (скорость оседания эритроцитов):  
   - **Норма**: до 10–15 мм/ч (зависит от возраста и пола)  
   - **При воспалении**: > 20–30 мм/ч (может быть 50–100 мм/ч при гнойных процессах)  

### **4. Токсическая зернистость нейтрофилов** (при тяжелых бактериальных инфекциях)  

### **5. Возможные сопутствующие изменения**:  
   - **Анемия** (снижение гемоглобина и эритроцитов при хроническом воспалении)  
   - **Тромбоцитоз** (увеличение тромбоцитов – PLT > 400 × 10⁹/л) как реакция на воспаление  

### **При вирусных инфекциях**:  
   - **Лейкопения** или нормальное количество лейкоцитов  
   - **Лимфоцитоз** (↑ лимфоцитов)  
   - **Моноцитоз** (↑ моноцитов)  

### **При аллергических/паразитарных воспалениях**:  
   - **Эозинофилия** (↑ эозинофилов > 5%)  

Эти изменения помогают отличить бактериальное воспаление от вирусного и оценить тяжесть процесса. Для уточнения причины воспаления дополнительно могут назначаться **СРБ, прокальцитонин, ферритин и другие маркеры**.


In [ ]:
ac_infalm = (
    data['rbc'] == get_ref(gender, age, 'rbc') &
    data['hgb'] == get_ref(gender, age, 'hgb') &
    data['mcv'] == get_ref(gender, age, 'mcv') &
    data['mchc'] == get_ref(gender, age, 'mchc') &
    data['soe'] > get_ref(gender, age, 'soe') &
    data['htc'] == get_ref(gender, age, 'htc') &
    data['wbc'] > get_ref(gender, age, 'wbc') &
    data['ne_abs'] > get_ref(gender, age, 'ne_abs') &
    data['ba_abs'] == get_ref(gender, age, 'ba_abs') &
    data['eo_abs'] == get_ref(gender, age, 'eo_abs') &
    data['ly_abs'] == get_ref(gender, age, 'ly_abs') &
    data['plt'] > get_ref(gender, age, 'plt')
)

## 3 - хроническое воспаление
В клиническом анализе крови (ОАК) хроническое воспаление может проявляться следующими изменениями:  

### **1. Анемия хронического заболевания (АХЗ)**  
- **Снижение гемоглобина (Hb)** – умеренная нормоцитарная или микроцитарная анемия.  
- **Низкий уровень сывороточного железа** (но ферритин в норме или повышен).  
- **Нормальный или повышенный уровень ферритина** (так как он является острофазовым белком).  

### **2. Изменения лейкоцитов**  
- **Нормальное или слегка повышенное количество лейкоцитов** (в отличие от острого воспаления, где выражен лейкоцитоз).  
- **Сдвиг лейкоцитарной формулы влево** (умеренное увеличение нейтрофилов, возможно появление незрелых форм).  
- **Лимфопения** (снижение лимфоцитов) при длительном воспалении.  

### **3. Тромбоцитоз**  
- **Умеренное повышение тромбоцитов** (тромбоцитоз) – реакция на хроническое воспаление.  

### **4. Ускоренная СОЭ**  
- **Повышенная скорость оседания эритроцитов (СОЭ)** – характерный, но неспецифический признак.  

### **5. Повышение острофазовых белков**  
- **Высокий уровень С-реактивного белка (СРБ)** – более чувствительный маркер, чем СОЭ.  
- **Повышенный фибриноген** (может влиять на СОЭ).  

### **Отличия от острого воспаления**  
- Нет резкого лейкоцитоза с нейтрофилезом.  
- Анемия развивается постепенно.  
- СОЭ и СРБ повышены, но не так резко, как при остром процессе.  

Эти изменения характерны для хронических инфекций (туберкулез, остеомиелит), аутоиммунных заболеваний (ревматоидный артрит, СКВ), онкологических процессов и других длительных воспалительных состояний. Для уточнения причины требуются дополнительные исследования (биохимия, иммунологические тесты, инструментальная диагностика).


In [ ]:
chr_infalm = (
    data['rbc'] == get_ref(gender, age, 'rbc') &
    data['hgb'] < get_ref(gender, age, 'hgb') &
    data['mcv'] == get_ref(gender, age, 'mcv') &
    data['mchc'] == get_ref(gender, age, 'mchc') &
    data['soe'] > get_ref(gender, age, 'soe') &
    data['htc'] == get_ref(gender, age, 'htc') &
    data['wbc'] == get_ref(gender, age, 'wbc') &
    data['ne_abs'] > get_ref(gender, age, 'ne_abs') &
    data['ba_abs'] == get_ref(gender, age, 'ba_abs') &
    data['eo_abs'] == get_ref(gender, age, 'eo_abs') &
    data['ly_abs'] == get_ref(gender, age, 'ly_abs') &
    data['plt'] > get_ref(gender, age, 'plt')
)

## 4 - специфическое воспаление
Специфическое воспаление (например, при туберкулезе, сифилисе, саркоидозе, некоторых грибковых инфекциях) может иметь **особые черты в клиническом анализе крови (КАК)**, отличающие его от обычного бактериального или вирусного воспаления.  

### **Основные признаки в ОАК:**  
1. **Умеренный лейкоцитоз или нормальное количество лейкоцитов**  
   - В отличие от гнойного воспаления (нейтрофильный лейкоцитоз со сдвигом влево) или вирусной инфекции (лимфоцитоз).  
   - Возможна **лейкопения** (например, при туберкулезе, ВИЧ-ассоциированных инфекциях).  

2. **Лимфоцитоз или моноцитоз**  
   - **Лимфоцитоз** характерен для хронических специфических процессов (туберкулез, бруцеллез).  
   - **Моноцитоз** (>10%) – маркер гранулематозных заболеваний (туберкулез, саркоидоз, сифилис).  

3. **Эозинофилия**  
   - Может встречаться при паразитарных инфекциях, грибковых поражениях, некоторых формах туберкулеза.  

4. **Ускоренное СОЭ**  
   - Часто значительно повышено (30–60 мм/ч), особенно при активном процессе.  

5. **Анемия хронического заболевания**  
   - Нормохромная или гипохромная анемия (снижение гемоглобина и эритроцитов).  
   - Повышение ферритина при нормальном или сниженном уровне железа.  

6. **Тромбоцитоз или тромбоцитопения**  
   - **Тромбоцитоз** – реактивный, как признак хронического воспаления.  
   - **Тромбоцитопения** – при тяжелых инфекциях (например, милиарный туберкулез).  

### **Дополнительные особенности при конкретных заболеваниях:**  
- **Туберкулез**: лимфоцитоз/моноцитоз, возможна анемия, высокая СОЭ.  
- **Саркоидоз**: моноцитоз, иногда эозинофилия, повышенная СОЭ.  
- **Сифилис**: лимфоцитоз, возможна анемия.  
- **Грибковые инфекции (гистоплазмоз, криптококкоз)**: эозинофилия, моноцитоз.  

### **Важно!**  
Изменения в ОАК неспецифичны и требуют **подтверждения дополнительными методами** (ПЦР, серология, биопсия, рентген/КТ). Для точной диагностики учитывают **клиническую картину, анамнез и другие анализы** (СРБ, прокальцитонин, иммунологические тесты).  

Если у вас есть конкретное заболевание – уточните, чтобы получить более детальную информацию.


In [ ]:
spec_infalm = (
    data['rbc'] == get_ref(gender, age, 'rbc') &
    data['hgb'] < get_ref(gender, age, 'hgb') &
    data['mcv'] == get_ref(gender, age, 'mcv') &
    data['mchc'] == get_ref(gender, age, 'mchc') &
    data['soe'] > get_ref(gender, age, 'soe') &
    data['htc'] == get_ref(gender, age, 'htc') &
    data['wbc'] == get_ref(gender, age, 'wbc') &
    data['ne_abs'] == get_ref(gender, age, 'ne_abs') &
    data['ba_abs'] == get_ref(gender, age, 'ba_abs') &
    data['eo_abs'] > get_ref(gender, age, 'eo_abs') &
    data['ly_abs'] > get_ref(gender, age, 'ly_abs') &
    data['plt'] > get_ref(gender, age, 'plt')
)

## 5 - аллергия
В клиническом анализе крови (ОАК) при аллергии могут наблюдаться следующие изменения:  

### **1. Эозинофилия**  
- **Повышение уровня эозинофилов** (норма: 1–5% от общего числа лейкоцитов, абсолютное количество – 0,02–0,3 × 10⁹/л).  
- При аллергии их уровень может достигать **5–15% и более**, особенно при тяжелых реакциях (например, при бронхиальной астме, атопическом дерматите).  

### **2. Лейкоцитоз (реже)**  
- Умеренное повышение общего числа лейкоцитов (норма: 4–9 × 10⁹/л), но это неспецифический признак.  

### **3. Базофилия**  
- **Увеличение базофилов** (норма: 0–1%), особенно при хронической аллергии или контакте с аллергеном.  

### **4. Лимфоцитоз (иногда)**  
- Повышение лимфоцитов (норма: 20–40%) может наблюдаться при аллергических реакциях, но это непостоянный признак.  

### **5. СОЭ (обычно в норме или слегка повышена)**  
- При аллергии без инфекции СОЭ редко превышает **15–20 мм/ч**.  

### **Важно!**  
Эти изменения **не являются строго специфичными** для аллергии и могут встречаться при:  
- Гельминтозах,  
- Аутоиммунных заболеваниях,  
- Опухолях,  
- Некоторых инфекциях (например, паразитарных).  

### **Дополнительные исследования для подтверждения аллергии:**  
- **Анализ на IgE общий** (повышен при атопии),  
- **Специфические IgE** (например, ImmunoCAP, аллергопанели),  
- **Кожные пробы** (прик-тесты).  

Если в ОАК выявлена эозинофилия, но аллергия не подтверждается, нужно исключать другие причины (паразиты, болезни крови и др.).


In [ ]:
spec_infalm = (
    data['rbc'] == get_ref(gender, age, 'rbc') &
    data['hgb'] == get_ref(gender, age, 'hgb') &
    data['mcv'] == get_ref(gender, age, 'mcv') &
    data['mchc'] == get_ref(gender, age, 'mchc') &
    data['soe'] == get_ref(gender, age, 'soe') &
    data['htc'] == get_ref(gender, age, 'htc') &
    data['wbc'] > get_ref(gender, age, 'wbc') &
    data['ne_abs'] == get_ref(gender, age, 'ne_abs') &
    data['ba_abs'] > get_ref(gender, age, 'ba_abs') &
    data['eo_abs'] > get_ref(gender, age, 'eo_abs') &
    data['ly_abs'] == get_ref(gender, age, 'ly_abs') &
    data['plt'] == get_ref(gender, age, 'plt')
)

## 6 - онко признаки
В клиническом анализе крови (КАК) при онкогематологических заболеваниях могут наблюдаться следующие изменения:  

### **1. Анемия**  
- **Снижение гемоглобина (Hb) и эритроцитов** (нормоцитарная или макроцитарная анемия).  
- **Ретикулоцитопения** (снижение молодых форм эритроцитов) – из-за угнетения эритропоэза.  
- **Анизоцитоз, пойкилоцитоз** (разные размеры и формы эритроцитов).  

### **2. Изменения лейкоцитов**  
- **Лейкоцитоз или лейкопения** (может быть как повышение, так и снижение).  
- **Бласты в периферической крови** (более 20% – признак острого лейкоза).  
- **Нейтропения** (снижение нейтрофилов) – частый признак при лейкозах и миелодиспластических синдромах.  
- **Лейкоцитарный «провал»** (отсутствие промежуточных форм – «hiatus leucemicus» при ОЛЛ или ОМЛ).  
- **Атипичные мононуклеары** (при лимфопролиферативных заболеваниях).  

### **3. Тромбоцитопения или тромбоцитоз**  
- **Снижение тромбоцитов** (часто при острых лейкозах, апластической анемии, МДС).  
- **Повышение тромбоцитов** (может быть при миелопролиферативных заболеваниях, например, при истинной полицитемии или эссенциальной тромбоцитемии).  

### **4. Другие признаки**  
- **Панцитопения** (снижение всех ростков крови) – при апластической анемии, МДС, инфильтрации костного мозга.  
- **Повышение СОЭ** (неспецифический признак, но часто при лимфомах, миеломной болезни).  
- **Тени Боткина-Гумпрехта** (полуразрушенные ядра лимфоцитов) – характерны для ХЛЛ.  

### **Подозрительные признаки на онкогематологию:**  
- **Цитопении неясного генеза** (особенно панцитопения).  
- **Появление бластов** в периферической крови.  
- **Высокий лейкоцитоз** с атипичными клетками.  
- **Тромбоцитоз** с другими изменениями (например, базофильно-эозинофильная ассоциация при ХМЛ).  

При обнаружении таких изменений требуется **консультация гематолога** и дополнительное обследование:  
- **миелограмма** (аспирация костного мозга),  
- **иммунофенотипирование**,  
- **цитогенетика и молекулярно-генетические исследования**.  

Если у вас есть конкретные показатели анализа – можно уточнить интерпретацию.

In [ ]:
spec_infalm = (
    data['rbc'] < get_ref(gender, age, 'rbc') &
    data['hgb'] < get_ref(gender, age, 'hgb') &
    data['mcv'] < get_ref(gender, age, 'mcv') &
    data['mchc'] < get_ref(gender, age, 'mchc') &
    data['soe'] > get_ref(gender, age, 'soe') &
    data['htc'] < get_ref(gender, age, 'htc') &
    data['wbc'] < get_ref(gender, age, 'wbc') &
    data['ne_abs'] < get_ref(gender, age, 'ne_abs') &
    data['ba_abs'] < get_ref(gender, age, 'ba_abs') &
    data['eo_abs'] < get_ref(gender, age, 'eo_abs') &
    data['ly_abs'] < get_ref(gender, age, 'ly_abs') &
    data['plt'] < get_ref(gender, age, 'plt')
)

In [ ]:
cat_list = ['gender', 'plasma', 'myelo', 'yunye', 'blasty',
            'prolym', 'promyelo', 'normoblast', 'hypochromia',
            'macrocytos', 'microcytos', 'anisocytos', 'poikilocytos',]
num_list = [col for col in data.columns if col not in cat_list]
data[cat_list] = data[cat_list].astype('object')

In [ ]:
print('Проверка тренировочной выборки:'+'\n'+'-' * 30)
check_data_quality(data)

In [ ]:
# data.drop(columns=[
#     'ret_abs',
#     # 'hypochromia', 'macrocytos', 'microcytos',
#     # 'poikilocytos', 'plasma', 'plcr', 'normoblast',
#     # 'prolym', 'promyelo'
# ], inplace=True)

In [ ]:
def outliers_iqr(data: pd.DataFrame, feature: str, left: float=1.5, 
                 rigth: float=1.5, log_scale: bool=False) -> tuple:
    """Функция расчитывает выбросы по межквартильному размаху (метод Тьюки)
    и возвращает кортеж с двумя датафреймами - чистый и с выбросами"""
    if log_scale:
        x = np.log(data[feature])
    else:
        x = data[feature]
    quartile_1, quartile_3 = x.quantile(0.25), x.quantile(0.75),
    iqr = quartile_3 - quartile_1
    lower_bound = quartile_1 - (iqr * left)
    upper_bound = quartile_3 + (iqr * rigth)
    outliers = data[(x < lower_bound) | (x > upper_bound)]
    cleaned = data[(x >= lower_bound) & (x <= upper_bound)]
    return outliers, cleaned

def outliers_z_score(data: pd.DataFrame, feature: str, left: float=3, 
                     right: float=3, log_scale: bool=False) -> tuple:
    """Функция расчитывает выбросы методом сигм (Z - отклонений)
    и возвращает кортеж с двумя датафреймами - чистый и с выбросами"""
    if log_scale:
        x = np.log(data[feature])
    else:
        x = data[feature]
    mu = x.mean()
    sigma = x.std()
    lower_bound = mu - left * sigma
    upper_bound = mu + right * sigma
    outliers = data[(x < lower_bound) | (x > upper_bound)]
    cleaned = data[(x >= lower_bound) & (x <= upper_bound)]
    return outliers, cleaned

In [ ]:
# for el in num_list:
#     fig, axes = plt.subplots(nrows=1, ncols=2, figsize=(15, 4))
#     histplot = sns.histplot(data=data, x=el, ax=axes[0], bins=25)
#     histplot.axvline(data[el].mean(), color='r', lw=1)
#     histplot.axvline(data[el].mean()+ 3 * data[el].std(), color='g', ls='--', lw=1)
#     histplot.axvline(data[el].mean()- 3 * data[el].std(), color='g', ls='--', lw=1)
#     boxplot = sns.boxplot(data=data, x=el, ax=axes[1]);

In [ ]:
for el in num_list:
    print(f'\n{'-' * 50}\nВыбросы признака {el}\n' + '-' * 50)
    outliers_irq, cleaned_irq = outliers_iqr(data, el)
    print(f'Число выбросов по методу Тьюки: {outliers_irq.shape[0]}')
    print(f'Результирующее число записей: {cleaned_irq.shape[0]}')
    print()
    outliers_z, cleaned_z = outliers_z_score(data, el)
    print(f'Число выбросов по методу z-отклонения: {outliers_z.shape[0]}')
    print(f'Результирующее число записей: {cleaned_z.shape[0]}')

    data = cleaned_irq

In [ ]:
# for el in num_list:
#     fig, axes = plt.subplots(nrows=1, ncols=2, figsize=(15, 4))
#     histplot = sns.histplot(data=data, x=el, ax=axes[0], bins=25)
#     histplot.axvline(data[el].mean(), color='r', lw=1)
#     histplot.axvline(data[el].mean()+ 3 * data[el].std(), color='g', ls='--', lw=1)
#     histplot.axvline(data[el].mean()- 3 * data[el].std(), color='g', ls='--', lw=1)
#     boxplot = sns.boxplot(data=data, x=el, ax=axes[1]);

In [ ]:
# # Создаем тепловую карту пропусков
# plt.figure(figsize=(10, 6))
# sns.heatmap(data.isnull(), cbar=False, cmap='viridis', yticklabels=False)
# plt.title('Тепловая карта пропущенных значений', fontsize=16)
# plt.show()

In [ ]:
# corr_matrix_df = round(data.corr(), 3)

# fig, ax = plt.subplots(figsize=(65, 60))
# sns.heatmap(
#     corr_matrix_df,
#     annot=True,
#     cmap='RdYlGn',
#     annot_kws={"fontsize": 18},
#     xticklabels=corr_matrix_df.columns,
#     yticklabels=corr_matrix_df.index,
#     linewidths=.5
# )
# ax.tick_params(axis="x", labelrotation=45, labelsize=14)
# ax.tick_params(axis="y", labelsize=14)
# plt.tight_layout()
# plt.show()

In [ ]:
corr_matrix_df = round(data.corr(), 3)

#выведем пары с корреляцией выше 0.5
threshold = 0.5

high_correlations = (
    corr_matrix_df
    .stack()
    .reset_index()
    .rename(columns={0:'Корреляция'})
    .query('level_0 != level_1')
    .query(f'abs(Корреляция) >= {threshold}')
)

result = high_correlations.sort_values(by='Корреляция', ascending=False)[['level_0', 'level_1', 'Корреляция']]
display(result)

In [ ]:
data.drop(columns=['hct', 'rbc', 'plasma', 'yunye', 'poikilocytos'], inplace=True)

In [ ]:
corr_matrix_df = round(data.corr(), 3)

#выведем пары с корреляцией выше 0.5
threshold = 0.5

high_correlations = (
    corr_matrix_df
    .stack()
    .reset_index()
    .rename(columns={0:'Корреляция'})
    .query('level_0 != level_1')
    .query(f'abs(Корреляция) >= {threshold}')
)

result = high_correlations.sort_values(by='Корреляция', ascending=False)[['level_0', 'level_1', 'Корреляция']]
display(result)

In [ ]:
print('Проверка тренировочной выборки:'+'\n'+'-' * 30)
check_data_quality(data)